In [7]:
# Install required libraries
!pip install transformers pandas torch openpyxl

# Import libraries
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW

# Load the labeled dataset
file_path = "/content/drive/My Drive/Colab Notebooks/lyrics_with_labels.xlsx"  # Update path if using Google Drive
df = pd.read_excel(file_path)

# Extract lyrics and labels
lyrics = df["Lyric"].dropna().tolist()
labels = df["Label"].dropna().tolist()

# Clean the text (optional if already cleaned in your labeled dataset)
def clean_text(text):
    import re
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove special characters
    return text

lyrics = [clean_text(lyric) for lyric in lyrics]

# Split dataset into training, validation, and testing sets
from sklearn.model_selection import train_test_split
train_lyrics, temp_lyrics, train_labels, temp_labels = train_test_split(
    lyrics, labels, test_size=0.3, random_state=42
)

val_lyrics, test_lyrics, val_labels, test_labels = train_test_split(
    temp_lyrics, temp_labels, test_size=0.5, random_state=42
)

# Tokenize lyrics for each set
tokenizer = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
train_inputs = tokenizer(
    train_lyrics, truncation=True, padding=True, max_length=128, return_tensors="pt"
)
val_inputs = tokenizer(
    val_lyrics, truncation=True, padding=True, max_length=128, return_tensors="pt"
)
test_inputs = tokenizer(
    test_lyrics, truncation=True, padding=True, max_length=128, return_tensors="pt"
)

# Convert labels to tensors
train_labels_tensor = torch.tensor(train_labels, dtype=torch.float32)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.float32)

# Create a PyTorch dataset
class LyricsDataset(Dataset):
    def __init__(self, tokenized_inputs, labels):
        self.inputs = tokenized_inputs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.inputs["input_ids"][idx],
            "attention_mask": self.inputs["attention_mask"][idx],
            "labels": self.labels[idx]
        }

# Create datasets and dataloaders
train_dataset = LyricsDataset(train_inputs, train_labels_tensor)
val_dataset = LyricsDataset(val_inputs, val_labels_tensor)
test_dataset = LyricsDataset(test_inputs, test_labels_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Define model and optimizer
model = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=1)
optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop with validation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(3):  # Number of epochs
    model.train()
    total_loss = 0
    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Training Loss: {total_loss / len(train_dataloader)}")

    # Validation step
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()

    print(f"Epoch {epoch}, Validation Loss: {val_loss / len(val_dataloader)}")

# Evaluate on testing set
model.eval()
test_loss = 0
predictions = []
with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        test_loss += outputs.loss.item()

        preds = outputs.logits.detach().cpu().numpy()
        predictions.extend(preds)

print(f"Test Loss: {test_loss / len(test_dataloader)}")
print("Example predictions:", predictions[:5])  # Display first 5 predictions

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0, Training Loss: 0.1672599169318206
Epoch 0, Validation Loss: 0.07863728736992925
Epoch 1, Training Loss: 0.09536546791115633
Epoch 1, Validation Loss: 0.08461096479247014
Epoch 2, Training Loss: 0.08573644035137616
Epoch 2, Validation Loss: 0.08096611499786377
Test Loss: 0.1073796699444453
Example predictions: [array([0.6136664], dtype=float32), array([0.5820112], dtype=float32), array([0.53670615], dtype=float32), array([0.58945274], dtype=float32), array([0.55645394], dtype=float32)]
